# 04 — The Complete RAG Pipeline

This notebook ties everything together: load → chunk → embed → store → retrieve → generate.

**What you'll learn:**
- Using the `RAGPipeline` class for end-to-end question answering
- Inspecting what the LLM sees with `preview()`
- Getting cited answers from your knowledge base
- Debugging retrieval vs. generation failures

**Prerequisite:** You need an API key from OpenAI or Anthropic in your `.env` file.

## 1. Setup

In [ ]:
import sys, os
os.chdir(os.path.join(os.path.dirname(os.path.abspath('.')), ''))
sys.path.insert(0, 'src')

from dotenv import load_dotenv
load_dotenv()

# Check which API keys are available
has_openai = bool(os.getenv('OPENAI_API_KEY'))
has_anthropic = bool(os.getenv('ANTHROPIC_API_KEY'))

print(f'OpenAI key:    {"found" if has_openai else "NOT FOUND"}')
print(f'Anthropic key: {"found" if has_anthropic else "NOT FOUND"}')

if not has_openai and not has_anthropic:
    print('\n⚠ No API key found! Copy .env.example to .env and add your key.')
    print('  You can still run preview() to see the prompt without an API call.')

## 2. Initialize the pipeline

In [ ]:
from rag_pipeline.pipeline import RAGPipeline

# Choose your provider based on which key you have
provider = 'openai' if has_openai else 'anthropic'

pipe = RAGPipeline(
    persist_dir='./chroma_db',
    llm_provider=provider,
)

print(f'Pipeline initialized with {provider}')
print(f'Collection stats: {pipe.stats()}')

## 3. Index the sample documents

In [ ]:
result = pipe.index('data/sample/')
print(f'Indexing complete:')
for key, val in result.items():
    print(f'  {key}: {val}')

## 4. Preview the prompt (no API call needed)

Before spending API credits, inspect what the LLM would see.

In [ ]:
print(pipe.preview('What is the remote work policy?'))

## 5. Inspect retrieval separately

Check which chunks are retrieved before calling the LLM.

In [ ]:
results = pipe.retrieve('What is the remote work policy?', top_k=5)

print(f'Retrieved {len(results)} chunks:\n')
for i, r in enumerate(results, 1):
    sim = 1 - r['distance']
    source = r['metadata']['source'].split('/')[-1]
    chunk = r['metadata'].get('chunk_index', '?')
    print(f'#{i} [{sim:.3f}] {source}, chunk {chunk}')
    print(f'   {r["document"][:100]}...\n')

## 6. Ask questions — the full RAG pipeline

**This cell requires an API key.**

In [ ]:
result = pipe.query('What is the remote work policy?')

print(f'Model: {result["model"]}')
print(f'Sources: {result["num_results"]} chunks retrieved\n')
print('Answer:')
print(result['answer'])
print('\nSources used:')
for s in result['sources']:
    print(f'  - {s["source"]} chunk {s["chunk_index"]} '
          f'(similarity: {s["similarity"]})')

## 7. Multiple questions across both documents

In [ ]:
questions = [
    'How much PTO do employees with 4 years of tenure get?',
    'What is the code review process?',
    'What happens during a production deployment?',
    'What is the 401k matching policy?',
    'What are the security best practices for authentication?',
]

for q in questions:
    result = pipe.query(q, top_k=3)
    print(f'\n{"=" * 60}')
    print(f'Q: {q}')
    print(f'A: {result["answer"]}')
    sources = [f'{s["source"]}:{s["chunk_index"]}' for s in result['sources']]
    print(f'Sources: {sources}')

## 8. Graceful failure — out-of-scope questions

A good RAG system admits when it doesn't know.

In [ ]:
out_of_scope = [
    'How do I bake a chocolate cake?',
    'What is the capital of France?',
    'Explain quantum entanglement.',
]

for q in out_of_scope:
    result = pipe.query(q, top_k=3)
    print(f'Q: {q}')
    print(f'A: {result["answer"]}\n')

## 9. Experiment: top_k = 2 vs 5

Does more context help or hurt?

In [ ]:
question = 'What benefits does the company offer?'

for k in [2, 5]:
    result = pipe.query(question, top_k=k)
    print(f'\n--- top_k={k} ---')
    print(f'Answer ({len(result["answer"])} chars):')
    print(result['answer'])
    print(f'Sources: {[s["source"] + ":" + str(s["chunk_index"]) for s in result["sources"]]}')

## Key Takeaways

1. **The RAG pipeline is: embed query → retrieve chunks → build prompt → call LLM**
2. **`preview()`** lets you inspect the prompt without an API call — essential for debugging
3. **`retrieve()`** lets you check retrieval quality independently of generation
4. **Most bad answers are retrieval failures** — check retrieval first, generation second
5. **Good RAG systems fail gracefully** — they say "I don't know" instead of hallucinating

**Next:** We'll improve retrieval quality with hybrid search and reranking.